# **Homework 2: Improving RAG with Wikidata Information Retrieval: an In-Depth Analysis**

In this notebook, we present the full runnable pipeline for the **Homework 2: Improving RAG with Wikidata Information Retrieval: an
In-Depth Analysis** report.

We first initialize the environment by installing and importing all the necessary depndencies and libraries.

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import json
import nltk
import numpy as np
import os
import pandas as pd
import random
import re
import requests
import sklearn.metrics
import sys
import torch

from datasets import Dataset, load_dataset
from nltk.translate.meteor_score import meteor_score
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from urllib3.util.retry import Retry
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## Custom Functions Initalization

Before proceeding to the effective pipeline, we initalize all the custom functions that have been created.

### Generic Utils

In [ ]:
# Fixing the seed for reproducibility of specific pipelines
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# Loading the dataset and splitting it
def load_data():
    ds = load_dataset(
        "sapienzanlp-course-materials/hw-mnlp-2026"
    )

    ds_train = ds["train"]
    train_dev_split = ds["train"].train_test_split(test_size=0.1, seed=42)
    ds_train = train_dev_split["train"]
    ds_dev = train_dev_split["test"]

    return {
        "train": ds_train,
        "dev": ds_dev,
        "test": ds["test"],
        "blind": ds["blind"]
    }

In [ ]:
# Model loading function according to its type
def load_model(model, model_type):
    tokenizer = AutoTokenizer.from_pretrained(model)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Differentiate the configuration between GPU/CPU processing and Causal/Seq2Seq models
    if device == 'cuda':
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16, # Accelerate computation
            bnb_4bit_quant_type="nf4",            # Best compression quality
            bnb_4bit_use_double_quant=True        # VRAM economy
        )
        if model_type == "causal":
            model = AutoModelForCausalLM.from_pretrained(model, quantization_config=bnb_config, device_map="auto")
        elif model_type == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(model, quantization_config=bnb_config, device_map="auto")
        device = next(model.parameters()).device

    else:
        if model_type == "causal":
            model = AutoModelForCausalLM.from_pretrained(model)
        elif model_type == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(model)
        model.to(device)

    return model, tokenizer, device

### JSONL and Excel Files Generation Functions

In [ ]:
# Create a jsonl file according to its type (judge evaluation or answer generation by LLM)
def create_jsonl_file(results, filepath, type):
    with open(filepath, 'w', encoding='utf-8') as f:
        for result in results:
            if type == "generated_responses":
                json_record = {
                    "query_id": result["query_id"],
                    "augmented_prompt": result["augmented_prompt"] if result.get("augmented_prompt") is not None else "",
                    "retrieved_chunks": result["retrieved_chunks"] if result.get("retrieved_chunks") is not None else [],
                    "generated_answer": result["generated_answer"]
                }
            elif type == "LLM_judge":
                json_record = {
                    "query_id": result["query_id"],
                    "retrieved_chunks": result["retrieved_chunks"],
                    "augmented_prompt": result["augmented_prompt"],
                    "generated_answer": result["generated_answer"],
                    "llm_judge": result["llm_judge_output"],
                    "annotator_1": result["annotator_1"],
                    "annotator_2": result["annotator_2"]
                }

            f.write(json.dumps(json_record, ensure_ascii=False) + '\n')

In [ ]:
# Generate jsonl file with correct name
def generate_jsonl_file(results, split_name, model_name, setting, type):
    group_name = "Its_always_loss"

    output_dir = os.path.join("rag", split_name)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    filename = f"{group_name}-{split_name}-{model_name}-{setting}.jsonl"
    filepath = os.path.join(output_dir, filename)
    print(f"{filename} generation...")
    create_jsonl_file(results, filepath, type)

In [ ]:
# function which takes in input the jsonl file generated during the generation of judgements and create an Excel file.
# This Excel file is used for the annotations
def export_judge_to_excel(jsonl_filepath, excel_filepath, queries, short_answers):
    records = []

    with open(jsonl_filepath, 'r', encoding='utf-8') as f:
        for line, query, short_answer in zip(f, queries, short_answers):
            if line.strip():
                record = json.loads(line)
                row = {
                    "query_id": record.get("query_id", ""),
                    "query": query,
                    "retrieved_chunks": str(record.get("retrieved_chunks", [])),
                    "augmented_prompt": record.get("augmented_prompt", ""),
                    "generated_answer": record.get("generated_answer", ""),
                    "short_answer": short_answer[0],
                    "llm_judge": record.get("llm_judge", 0),
                    "annotator_1": "",
                    "annotator_2": ""
                }
                records.append(row)

    df = pd.DataFrame(records)

    df.to_excel(excel_filepath, sheet_name="Jugements", index=False)

    print(f"excel file saved : {excel_filepath}")

In [ ]:
# Take in input the excel file annotated and convert it into jsonl file
def export_annotations(excel_filepath, jsonl_filepath):
    df = pd.read_excel(excel_filepath, sheet_name="Jugements", engine='openpyxl')

    records = []
    for _, row in df.iterrows():
        record = {
            "query_id": row.get("query_id", ""),
            "retrieved_chunks": row.get("retrieved_chunks", ""),
            "augmented_prompt": row.get("augmented_prompt", ""),
            "generated_answer": row.get("generated_answer", ""),
            "llm_judge": row.get("llm_judge", 0),
            "annotator_1": row.get("annotator_1", ""),
            "annotator_2": row.get("annotator_2", "")
        }
        records.append(record)

    with open(jsonl_filepath, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"jsonl file saved : {jsonl_filepath}")

In [ ]:
# Fonction that compute the Cohen's Kappa metric between annotators and judge.
def final_evaluation(file):
    overall_a1_scores = []
    overall_a2_scores = []
    overall_judge_scores = []

    with open(file, "r") as f:
        for line in f:
            item = json.loads(line)
            a1_score = int(item.get("annotator_1", 0))
            a2_score = int(item.get("annotator_2", 0))
            judge_score = int(item.get("llm_judge", 0))
            overall_a1_scores.append(a1_score)
            overall_a2_scores.append(a2_score)
            overall_judge_scores.append(judge_score)

    avg_a1_score = np.mean(overall_a1_scores)
    avg_a2_score = np.mean(overall_a2_scores)
    avg_judge_score = np.mean(overall_judge_scores)

    print(f"Average model's score for annotator 1: {avg_a1_score:.4f}")
    print(f"Average model's score for annotator 2: {avg_a2_score:.4f}")
    print(f"Average model's score for LLM-as-a-Judge: {avg_judge_score:.4f}")
    print(f"Cohen's Kappa between annotators: {sklearn.metrics.cohen_kappa_score(overall_a1_scores, overall_a2_scores):.4f}")
    print(f"Cohen's Kappa between annotator 1 and LLM-as-a-Judge: {sklearn.metrics.cohen_kappa_score(overall_a1_scores, overall_judge_scores):.4f}")
    print(f"Cohen's Kappa between annotator 2 and LLM-as-a-Judge: {sklearn.metrics.cohen_kappa_score(overall_a2_scores, overall_judge_scores):.4f}")

### Wikidata Functions Implementation

This functions serve for both Wikidata Information Retrieval pipelines as explained in the report, in the RAG and Oracle systems implementation we rely on retrieving an additional custom document exploiting a search algorithm on Wikidata information related to an associated Wikidata ID per query. Wikidata has been also used for augmenting the gold answer context, to provide a less rigid analysis for the used automatic evaluation metrics. Here we provide:

*   The HTTP session to properly call Wikidata APIs;
*   A set of gold properties, manually selected from the top-100 used properties in the whole Wikidata database;
*   A list of custom functions to fetch entities, labels, descriptions, aliases and properties depending on the required task (RAG or Answer Augmentation).





In [ ]:
# Start the HTTP session to handle disconnections
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[502, 503, 504])
adapter = HTTPAdapter(max_retries=retries)
session.mount("http://", adapter)
session.mount("https://", adapter)

In [ ]:
# Gold properties: properties that are more likely to be meaningful in a QA task for a given entity (manually selected from top-100 used properties in Wikidata).
WIKIDATA_GOLD_PROPERTIES = {
    "P31": "instance of",
    "P248": "stated in",
    "P17": "country",
    "P585": "point in time",
    "P106": "occupation",
    "P21": "sex or gender",
    "P9259": "epoch",
    "P569": "date of birth",
    "P582": "end time",
    "P27": "country of citizenship",
    "P734": "family name",
    "P361": "part of",
    "P279": "subclass of",
    "P276": "location",
    "P1082": "population",
    "P19": "place of birth",
    "P570": "date of death",
    "P69": "educated at",
    "P155": "follows",
    "P156": "followed by",
    "P495": "country of origin",
    "P527": "has part(s)",
}

In [ ]:
# Fetch the Wikidata entity information for a given Wikidata ID, including label, description, aliases, and selected properties (if 'id_only=False').
def get_wikidata_entity(wikidata_ids, id_only=False):
    # Check if if the function receives a single Wikidata ID or more, and convert it to a list if necessary.
    if isinstance(wikidata_ids, str):
        wikidata_ids = [wikidata_ids]
    elif isinstance(wikidata_ids, set):
        wikidata_ids = list(wikidata_ids)

    # Call Wikidata API to fetch entity information for the given Wikidata IDs.
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(wikidata_ids),
        "format": "json",
        "languages": "en",
        "props": "labels|descriptions|aliases|claims" # Retrieve labels, descriptions, aliases, and claims (properties) for the entities.
    }
    headers = {
        "User-Agent": "NLPHomework Bot (lorito.1885657@studenti.uniroma1.it)"
    }

    response = session.get(url, params=params, headers=headers)
    response.raise_for_status()

    data = response.json()

    if "error" in data:
        print(f"Error fetching Wikidata entity: {data['error']}")
        return None

    # If 'id_only' is True, return a dictionary mapping Wikidata IDs to their labels.
    if id_only:
        fetched_dict = {}
        for wikidata_id in wikidata_ids:
            entity = data.get("entities", {}).get(wikidata_id, {})
            label = entity.get("labels", {}).get("en", {}).get("value", wikidata_id)
            fetched_dict[wikidata_id] = label
        return fetched_dict

    # Otherwise, return a detailed string with the entity's information and selected properties.
    wikidata_entity_info = ""

    for wikidata_id in wikidata_ids:
        # Get the entity's label, description, and aliases.
        entity = data.get("entities", {}).get(wikidata_id, {})
        label = entity.get("labels", {}).get("en", {}).get("value", "Unknown label")
        description = entity.get("descriptions", {}).get("en", {}).get("value", "Unknown description")
        aliases = entity.get("aliases", {}).get("en", [])
        aliases = [alias.get("value") for alias in aliases] if aliases else []

        # Add the basic information about the entity to the result string.
        wikidata_entity_info += f"Wikidata Information: {label} - {description}; Aliases: {', '.join(aliases) if aliases else 'None'}"

        # If 'id_only' is False, also fetch and include selected properties of the entity in the result string.
        if not id_only:
            properties = get_entity_properties(wikidata_id, entity)
            for prop in properties:
                if "entity" in prop:
                    wikidata_entity_info += f"; {prop['property']}: {prop['entity']}"
                elif "value" in prop:
                    wikidata_entity_info += f"; {prop['property']}: {prop['value']}"

    if response.status_code == 200:
        return wikidata_entity_info
    else:
        print(f"Failed to fetch Wikidata entity. Status code: {response.status_code}")
        return None

In [ ]:
# Get selected properties of a Wikidata entity, prioritizing "gold" properties (limit of retrived properties for each entity is set to 10 for bounding the context enrichment in any case).
def get_entity_properties(wikidata_id, entity):
    claims = entity.get("claims", {})
    selected_properties= []
    ids_to_fetch = set() # Set of ids that needs to be fetched later to retrieve their textual information (e.g., linked entities, non-gold properties, etc.).

    # We fetch properties until getting at most 10 gold ones.
    selected_properties, ids_to_fetch = fetch_property(claims, selected_properties, ids_to_fetch, needs_gold=True) # We first look for the gold properties that are most likely to contain the answer.

    # If there are less than 10 selected gold properties, we still look for other properties.
    if len(selected_properties) < 10:
        selected_properties, ids_to_fetch = fetch_property(claims, selected_properties, ids_to_fetch, needs_gold=False) # We look for additional properties beyond the gold ones to enrich the context.

    # If there are any IDs to fetch, we do so to retrieve their textual information.
    if ids_to_fetch:
        # We fetch the entities for all the linked entities ids to retrieve their elements.
        fetched_ids = get_wikidata_entity(list(ids_to_fetch), id_only=True)

        # We substitute the entity's or property's id with the fetched element.
        for prop in selected_properties:
            if "val_id" in prop and prop["val_id"] in ids_to_fetch:
                prop["entity"] = fetched_ids.get(prop["val_id"])

            if prop["property"] in fetched_ids:
                prop["property"] = fetched_ids[prop["property"]]

    return selected_properties

In [ ]:
# Fetch selected properties of a Wikidata entity (based on its claims) and return a list of dictionaries with the property name and its value (either an entity or a literal value).
def fetch_property(claims, selected_properties, ids_to_fetch, needs_gold=False):
    for prop_id, prop_value in claims.items():
        # We stop fetching properties if we have already selected 10 of them.
        if len(selected_properties) >= 10:
            break
        # Check if the property is a gold property (if 'needs_gold' is True) or a non-gold property (if 'needs_gold' is False) and fetch its value accordingly.
        if (needs_gold and prop_id in WIKIDATA_GOLD_PROPERTIES) or (not needs_gold and prop_id not in WIKIDATA_GOLD_PROPERTIES):
            snak = claims[prop_id][0].get("mainsnak", {}) # We take the first mainsnak of the property, as it usually contains the most relevant value for our task (mainsnak is the tuple (property_id, value) of a property).
            if snak.get("snaktype") == "value":
                datatype = snak.get("datatype")

                # Avoids fetching properties with datatypes that are not properly meaningful in textual context (e.g., media, URLs, geo-shapes, etc.).
                if datatype in ["commonsMedia", "external-id", "url", "math", "geo-shape", "tabular-data", "musical-notation"]:
                    continue

                prop_name = WIKIDATA_GOLD_PROPERTIES.get(prop_id, prop_id) # If it's a gold property, we use its name, otherwise we keep the id as name.
                if prop_id not in WIKIDATA_GOLD_PROPERTIES:
                    ids_to_fetch.add(prop_id)

                val_type = snak.get("datavalue", {}).get("type")
                val_data = snak.get("datavalue", {}).get("value")

                # Add to 'ids_to_fetch' the id of the value if it's an entity.
                if val_type == "wikibase-entityid":
                    val_id = val_data.get("id")
                    selected_properties.append({"property": prop_name, "val_id": val_id})
                    ids_to_fetch.add(val_id)
                else:
                    # If it's a literal value, we check if it's a dictionary with a specific key (e.g., "amount", "text", "time") and we extract the relevant information or we convert it to string.
                    if isinstance(val_data, dict):
                        if "amount" in val_data:
                            val_data = val_data["amount"].replace("+", "")
                        elif "text" in val_data:
                            val_data = val_data["text"]
                        elif "time" in val_data:
                            val_data = val_data["time"].split("T")[0].replace("+", "")
                        else:
                            val_data = str(val_data)
                    selected_properties.append({"property": prop_name, "value": val_data})

    return selected_properties, ids_to_fetch

In [ ]:
# Get hypothetical short answer's aliases and synonyms to expand EM (or subEM)
def get_wikidata_ground_truth(wikidata_id, short_answer):
    answers = set()
    answers.add(short_answer.strip().lower())

    # Call Wikidata API to fetch entity information for the given Wikidata IDs.
    url = "https://www.wikidata.org/w/api.php"
    headers = {
        "User-Agent": "NLPHomework Bot (lorito.1885657@studenti.uniroma1.it)"
    }

    params = {
        "action": "wbgetentities",
        "ids": wikidata_id,
        "format": "json",
        "languages": "en",
        "props": "claims" # Retrieve properties only.
    }

    response = session.get(url, params=params, headers=headers)
    response.raise_for_status()

    data = response.json()

    if "error" in data:
        print(f"Error fetching Wikidata entity: {data['error']}")
        return None

    entity = data.get("entities", {}).get(wikidata_id, {}) # Get wikidata entity of the wikidata_id releated to given short answer

    properties = get_entity_properties(wikidata_id, entity) # Get wikidata entity's properties

    match = None

    # Check if any property is a match with the short answer
    for prop in properties:
        if "entity" in prop:
            if  short_answer.strip().lower() in prop["entity"].strip().lower() or prop["entity"].strip().lower() in short_answer.strip().lower():
                match = prop.get("val_id")
                break

    # If there is a match, find all aliases and labels of the matched entity
    if match:
        params = {
            "action": "wbgetentities",
            "ids": match,
            "format": "json",
            "languages": "en",
            "props": "labels|aliases"
        }

        response = session.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        entity = data.get("entities", {}).get(match, {})

        # Pick the labels and add them to the synonyms set (if not already included)
        label = entity.get("labels", {}).get("en", {}).get("value", None)
        if label and label not in answers:
            answers.add(label.strip().lower())

        # Pick the aliases and add them to the synonyms set (if not already included)
        aliases = entity.get("aliases", {}).get("en", [])
        for alias in aliases:
            alias_value = alias.get("value", "").strip().lower()
            if alias_value and alias_value not in answers:
                answers.add(alias_value)

    return answers

### LLM Generation Pipeline Functions

In this section, we provide all the custom functions for starting the generative pipelines: Baseline, RAG and Oracle. A fourth function get_top_k_chunks is provided in order to call the All-Mini-L6-v2 semantic search system to retrieve the top-3 chunks for each query for both RAG and Oracle pipelines.

In [ ]:
# Baseline Pipeline function: load tokenizer's model, generate the answer for the given query and return it
def baseline(model, tokenizer, query, device):
    prompt = f"Question: {query}\nAnswer: "

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    answer = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7, # Lower temperature for more focused responses
        top_p=0.9, # Use nucleus sampling to keep the most probable tokens (not fixed)
        do_sample=True,
        num_beams=1, # No beam search needed as we need a "strict" answer
        pad_token_id=tokenizer.eos_token_id
    )

    generated_answer = tokenizer.decode(answer[0], skip_special_tokens=True, verbose=False)

    # Extract the answer if it is included in a Key-Value structure in the text
    if "Answer:" in generated_answer:
        extracted_answer = generated_answer.split("Answer:")[-1].strip()
    else:
        extracted_answer = generated_answer

    return extracted_answer


In [ ]:
# Return the top k chunks and their corresponding indices from the jsonl file (the retriever from HW1)
def get_top_k_chunks(query_id, jsonl_path, candidate_chunks, k=3):
    top_k_indices = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            if query_id in data:
                top_k_indices = data[query_id][:k]
                break

    top_k_chunks = []
    for i in top_k_indices:
        top_k_chunks.append(candidate_chunks[i])

    return top_k_chunks, top_k_indices


In [ ]:
# RAG function : Create prompt with context, tokenize and generate answer for the given query. Return the answer and the final prompt sent to the model
def rag(model, tokenizer, query, retrieved_passages, device, wikidata_info=None, verbose=False):
    context = []

    # Format the passage for being clearer in the prompt
    for index, passage in enumerate(retrieved_passages):
        formatted_passage = f"Document {index+1}: {passage}"
        context.append(formatted_passage)

    # add wikidata information if exists
    if wikidata_info:
        context.append(wikidata_info)

    context = "\n---\n".join(context) # concatenation of retrieved passages

    prompt = (f"You are an expert in question answering. Given a set of retrieved documents, and information about a Wikidata entity, provide a concise and correct answer to the question.\n\n"
             f"Context: {context}\n"
             f"Question: {query}\n" # The question is being inserted after the context to avoid "Lost in the middle" issues
             f"Answer: ")
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length= tokenizer.model_max_length if tokenizer.model_max_length >= 1536 else 1536).to(device)

    # prompt human-readable (used to check if the truncation is not too important and if the query is in the final prompt)
    truncated_prompt = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)

    if verbose:
        print(f"\n--- FINAL PROMPT SENT TO THE MODEL {model.__class__.__name__} ---")
        print(truncated_prompt)
        print("-------------------------------------------------------------------\n")

    answer = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,# Lower temperature for more focused responses
        top_p=0.9, # Use nucleus sampling to keep the most probable tokens (not fixed)
        do_sample=True,
        num_beams=1, # No beam search needed as we need a "strict" answer
        pad_token_id=tokenizer.eos_token_id
    )

    generated_answer = tokenizer.decode(answer[0], skip_special_tokens=True)

    # Delete the "Answer: " word if present in the generated answer
    if "Answer:" in generated_answer:
        extracted_answer = generated_answer.split("Answer:")[-1].strip()
    else:
        extracted_answer = generated_answer

    return extracted_answer, truncated_prompt


In [ ]:
# Oracle function : put the gold chunk in first position of the retrieved chunks. Return the updated chunk list and their indices.
def oracle(retrieved_chunks, retrieved_indices, gold_index, candidates):
    indices_list = list(retrieved_indices)
    chunks_list = list(retrieved_chunks)

    if gold_index in indices_list:
        # put it at first (index 0)
        idx_in_list = indices_list.index(gold_index)
        gold_idx = indices_list.pop(idx_in_list)
        gold_text = chunks_list.pop(idx_in_list)
        indices_list.insert(0, gold_idx)
        chunks_list.insert(0, gold_text)
    else:
        # delete the last one
        indices_list.pop(-1)
        chunks_list.pop(-1)
        # add the gold one at first
        indices_list.insert(0, gold_index)
        chunks_list.insert(0, candidates[gold_index])

    return chunks_list, indices_list


### Evaluation Functions

In this section, we provide all the necessary functions to call the automatic evaluation metrics (EM, subEM, METEOR). There is also a function to provide the average scores for all metrics, for both models and their generative pipelines, both with or without Wikidata Answers Augmentation.

In [ ]:
# Exact match here means the strictly same answer (no additionnal word)
def compute_exact_match(prediction, ground_truths):
    for ground_truth in ground_truths:
        if prediction.strip().lower() == ground_truth.strip().lower():
            return 1
    return 0

In [ ]:
# sub Exact Match means the real answer is included in the generated answer
def compute_sub_EM(prediction, ground_truths):
    for ground_truth in ground_truths:
        if ground_truth.strip().lower() in prediction.strip().lower():
            return 1
    return 0

In [ ]:
# METEOR metric : compute a mix of Exact Match, Stem Match, Paraphrase Match, Synonym Match and n-grams overlap.
def compute_meteor(prediction, ground_truths):
    scores = []
    prediction_tokens = nltk.word_tokenize(prediction.lower())
    for ground_truth in ground_truths:
        ground_truth_tokens = nltk.word_tokenize(ground_truth.lower())
        scores.append(meteor_score([ground_truth_tokens], prediction_tokens))
    return max(scores) if scores else 0

In [ ]:
# evaluate the generated answer on the 3 metrics
def evaluate_all(prediction, ground_truth):
    return {
        "EM" : (compute_exact_match(prediction, ground_truth)),
        "subEM" : (compute_sub_EM(prediction, ground_truth)),
        "METEOR" : (compute_meteor(prediction, ground_truth))
    }


In [ ]:
def evaluate_generations(model, query_ids, short_answers, wikidata_answers, system ,file):
    gold_answers = dict(zip(query_ids, short_answers))

    # creation of a new file that contains the results of the different metrics
    output_file = file.replace(".jsonl", "_scored.jsonl")

    results = []
    total_scores = []
    total_wikidata_scores = []

    with open(file, "r") as f:
        for line in f:
            item = json.loads(line)
            query_id = item["query_id"]
            generated_answer = item["generated_answer"]

            short_answer = gold_answers.get(query_id, "")
            # compute the score without wikidata data
            scores = evaluate_all(generated_answer, short_answer)
            # compute the score with additional wikidata data
            wikidata_scores = evaluate_all(generated_answer, wikidata_answers[query_ids.index(query_id)])

            total_scores.append(scores)
            total_wikidata_scores.append(wikidata_scores)

            item["scores"] = scores
            item["wikidata_scores"] = wikidata_scores
            results.append(item)

    with open(output_file, "w") as f:
        for item in results:
            f.write(json.dumps(item) + "\n")

    # compute the average of each metric, with and without data from wikidata
    for metric in ["EM", "subEM", "METEOR"]:
        avg_score = sum(score[metric] for score in total_scores) / len(total_scores)
        avg_wikidata_score = sum(score[metric] for score in total_wikidata_scores) / len(total_wikidata_scores)
        print(f"Average {metric} for {model} on {system}: {avg_score:.4f}")
        print(f"Average {metric} for {model} on {system} (using Wikidata answers as ground truth): {avg_wikidata_score:.4f}")

### LLM-as-a-Judge functions

In this section we provide the whole set of functions to initialize the LLM-as-a-Judge pipeline. We provide:
*   A function for defining prompt, judgement generation and score extraction;
*   A function for initializing the LLM-as-a-Judge model (Mistral-7B-v0.3), calling the judge_answers function defined above and setup the JSONL file generation;





In [ ]:
# Define LLM-as-a-Judge prompt, load the tokenizer, generate judgement and save the Judge's score
def judge_answers(evaluator, tokenizer, device, query, llm_answer, short_answer):
    prompt = ("You are an expert evaluator. Assess if the LLM Answer is correct based on the Short Answer (a Short Answer is a set of concise and correct answers)."
              "You must respond only with a valid JSON object containing two keys: 'reasoning' (a brief explanation) and 'score' (integer 1 or 0):\n" # These two keys are fundamental in order to force the LLM to separate its reasoning process to the effective score
              "- reasoning: a brief explanaton of why the LLM answer is correct or not based on the short answer.\n"
              "- score: 1 if the LLM answer is correct, 0 otherwise.\n"
              f"Query: {query}\n"
              f"LLM Answer: {llm_answer}\n"
              f"Short Answer: {short_answer}\n"
              "Evaluation:")

    # Load the tokenizer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    # Generate the answer
    generated_judgement = evaluator.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False, # Forcing rigid and factual answers
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the answer
    generated_judgement = tokenizer.decode(generated_judgement[0], skip_special_tokens=True).strip()

    # Find the given score: if no score has been assigned, return 0
    match = re.search(r'"score"\s*:\s*([01])', generated_judgement)
    if match:
        score = match.group(1)
    else:
        score = "0"

    return score

In [ ]:
# Load Mistral-7B-Instruct to be the LLM-as-a-judge, define the answer's JSONL file path, retrieve scores for the given queries and model's answers and save the result in the JSONL file
def judge_model(model, queries):
    judge_model, judge_tokenizer, judge_device = load_model("mistralai/Mistral-7B-Instruct-v0.3", "causal") # Load the model

    # Define JSON file path
    ANSWERS_DIR = os.path.join(parent_dir, "rag/all-test")
    model_answers_path = os.path.join(ANSWERS_DIR, f"Its_always_loss-all-test-{model}-RAG.jsonl")

    judgements = []
    model_answers = {}

    # Load model's answers for each query in the file
    with open(model_answers_path, "r") as f:
        for line in f:
            data = json.loads(line)
            model_answers[data["query_id"]] = data

    # For each query, call the judge_answers function
    for query, query_id, short_answer in tqdm(queries, desc="Judging Answers"):
        rag_item = model_answers.get(query_id)

        judgement = judge_answers(judge_model, judge_tokenizer, judge_device, query, rag_item['generated_answer'], short_answer)

        # Save results for JSONL generation
        result_dict = {
            "query_id": rag_item["query_id"],
            "retrieved_chunks": rag_item.get("retrieved_chunks", []),
            "augmented_prompt": rag_item.get("augmented_prompt", ""),
            "generated_answer": rag_item["generated_answer"],
            "llm_judge_output": judgement,
            "annotator_1": "",
            "annotator_2": ""
        }
        judgements.append(result_dict)

    return judgements

## Part 1: LLM Answers Generation Pipelines

In this first part, we setup and start all the three generative pipelines: Baseline, RAG and Oracle.

### Data and Model Loading

We first load the required dataset and the selected models. For this project, we use 2 Small LM :
- flan-T5-large: a 0.8B encoder-decoder model (seq2seq)
which has been finetuned on different tasks
using instruction finetuning, which consists of asking the model to answer/solve correctly a set of questions/tasks which have been phrased as instructions;
- Llama-3.2-1B-Instruct: a 1B decoder-only
model (causal) fine-tuned for instruction-following and dialogue use cases. It also excels in knowledge retrieval and summarization.

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
ds = load_data()

queries = ds["test"]["query"]
wikidata_ids = ds["test"]["wikidata_id"]

t5_model = "google/flan-t5-large"
t5_model, t5_tokenizer, t5_device = load_model(t5_model, "seq2seq") # Flan-T5 is a seq2seq model, so we specify "seq2seq" here

llama_model = "meta-llama/Llama-3.2-1b-instruct"
llama_model, llama_tokenizer, llama_device = load_model(llama_model, "causal") # LLaMA is a causal model, so we specify "causal" here

README.md:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

...

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

We also recover the JSONL file of the best performing model in the last assignment to use it will be used for RAG and Oracle pipelines for chunk retrieval.

In [ ]:
!mkdir -p predictions/test
!mkdir -p predictions/blind

url = 'https://drive.google.com/uc?id=1X38E9e_smWx3BZwmXmiTnICuVrqxSv2j'

!gdown {url} -O predictions/test/Its_always_loss-test-all-miniLM-L6-v2-2-mnr-cosine.jsonl
!gdown {url} -O predictions/blind/Its_always_loss-blind-all-miniLM-L6-v2-2-mnr-cosine.jsonl

parent_dir = os.getcwd()

Downloading...
From: https://drive.google.com/uc?id=1X38E9e_smWx3BZwmXmiTnICuVrqxSv2j
To: /content/predictions/test/Its_always_loss-test-all-miniLM-L6-v2-2-mnr-cosine.jsonl
100% 234k/234k [00:00<00:00, 5.48MB/s]
Downloading...
From: https://drive.google.com/uc?id=1X38E9e_smWx3BZwmXmiTnICuVrqxSv2j
To: /content/predictions/blind/Its_always_loss-blind-all-miniLM-L6-v2-2-mnr-cosine.jsonl
100% 234k/234k [00:00<00:00, 6.03MB/s]


### Baseline Pipeline

We now proceed with the baseline pipeline.

In [ ]:
# Baseline pipeline
answers_baseline = {}
scores_baseline = {}

t5_baseline_all_results = []
llama_baseline_all_results = []

# For each query, we define the baseline generation
for query in tqdm(queries, desc="Baseline Pipeline Processing"):
    item = ds["test"][queries.index(query)]
    query = item["query"]
    query_id = item["query_id"]
    candidate = item["candidate_chunks"]
    short_answer = item["short_answer"]

    # Call the baseline pipeline custom function
    t5_answer_baseline = baseline(t5_model, t5_tokenizer, query, t5_device)
    llama_answer_baseline = baseline(llama_model, llama_tokenizer, query, llama_device)

    answers_baseline[query] = f"1st Model Answer: {t5_answer_baseline}\n2nd Model Answer: {llama_answer_baseline}\nReal Answer: {short_answer}" # Store real answer

    # Apprend query and answer to results structure to generate the JSONL file later
    t5_baseline_all_results.append({
        "query_id": query_id,
        "generated_answer": t5_answer_baseline,
    })

    llama_baseline_all_results.append({
        "query_id": query_id,
        "generated_answer": llama_answer_baseline,
    })

print("------------Final Answers (Baseline):------------")
for query, answer in answers_baseline.items():
    print(f"Query: {query}\n{answer}\n")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


------------Final Answers (Baseline):------------
Query: where is the lady with an ermine located
1st Model Answer: San Francisco
2nd Model Answer: 1st Earl of Malmesbury

The Lady with an Ermine is a famous painting by Sir John Everett Millais, created in 1885. The painting depicts a lady (the Lady with an Ermine) in the midst of a
Real Answer: ['the National Museum in Kraków']

Query: lok sabha or rajya sabha which is bigger
1st Model Answer: Lok Sabha
2nd Model Answer: Lok Sabha is bigger than the Rajya Sabha.
 Lok Sabha (House of the People) is the lower house of the Indian Parliament and is the largest among the three houses of the Indian Parliament. It is the main chamber of the Indian Parliament and is
Real Answer: ['Lok Sabha']

Query: who was the actress who played pinky tuscadero on happy days
1st Model Answer: Janet Gaynor
2nd Model Answer: Mary Kay Bergman was the actress who played Pinky Tuscadero on Happy Days.
Real Answer: ['Roz Kelly']

Query: who wrote the song i want 

### RAG and Oracle Pipelines

We can now proceed with RAG and Oracle pipelines. We first call the retriever.

In [ ]:
PREDICTIONS_DIR = os.path.join(parent_dir, "predictions")
all_mini_jsonl = os.path.join(PREDICTIONS_DIR, "test", "Its_always_loss-test-all-miniLM-L6-v2-2-mnr-cosine.jsonl")

We now execute the whole RAG pipeline.

In [ ]:
answers_rag = {}
scores_rag = {}
answers_oracle = {}
scores_oracle = {}
t5_rag_all_results = []
llama_rag_all_results = []
t5_oracle_all_results = []
llama_oracle_all_results = []

# For each query, retrieve the associated wikidata entity, find the top-k chunks from semantic search, generate models' answers based on RAG and Oracle pipelines
for query in tqdm(queries, desc="RAG & Oracle Pipeline Procesing"):
    item = ds["test"][queries.index(query)]
    query = item["query"]
    query_id = item["query_id"]
    candidate = item["candidate_chunks"]
    gold_index = item["answer_pos"]
    short_answer = item["short_answer"]
    wikidata_id = item["wikidata_id"]

    wikidata_info = get_wikidata_entity(wikidata_id) # Get Wikidata entity

    # RAG Pipeline: retrieve top-k chunks and generate answers with RAG
    retrieved_chunks, retrieved_indices = get_top_k_chunks(query_id, all_mini_jsonl, candidate, k=3)

    t5_answer_rag, t5_augmented_prompt = rag(t5_model, t5_tokenizer, query, retrieved_chunks, t5_device, wikidata_info)
    llama_answer_rag, llama_augmented_prompt = rag(llama_model, llama_tokenizer, query, retrieved_chunks, llama_device, wikidata_info)

    answers_rag[query] = f"1st Model RAG Answer: {t5_answer_rag}\n2nd Model RAG Answer: {llama_answer_rag}\nReal Answer: {short_answer}"

    # Append query and answer to results structure to generate the JSONL file later
    t5_rag_all_results.append({
        "query_id": query_id,
        "retrieved_chunks": retrieved_indices,
        "augmented_prompt": t5_augmented_prompt,
        "generated_answer": t5_answer_rag,
    })

    llama_rag_all_results.append({
        "query_id": query_id,
        "retrieved_chunks": retrieved_indices,
        "augmented_prompt": llama_augmented_prompt,
        "generated_answer": llama_answer_rag,
    })

    # Oracle pipeline: retrieve top-k chunks promoting the "oracle" chunk and generate answers with RAG (no Wikidata information provided)
    retrieved_chunks_oracle, retrieved_indices_oracle = oracle(retrieved_chunks, retrieved_indices, gold_index, candidate)

    t5_answer_oracle, t5_oracle_augmented_prompt = rag(t5_model, t5_tokenizer, query, retrieved_chunks_oracle, t5_device)
    llama_answer_oracle, llama_oracle_augmented_prompt = rag(llama_model, llama_tokenizer, query, retrieved_chunks_oracle, llama_device)

    answers_oracle[query] = f"1st Model Oracle Answer: {t5_answer_oracle}\n2nd Model Oracle Answer: {llama_answer_oracle}\nReal Answer: {short_answer}"

    # Append query and answer to results structure to generate the JSONL file later
    t5_oracle_all_results.append({
        "query_id": query_id,
        "retrieved_chunks": retrieved_indices_oracle,
        "augmented_prompt": t5_oracle_augmented_prompt,
        "generated_answer": t5_answer_oracle,
    })

    llama_oracle_all_results.append({
        "query_id": query_id,
        "retrieved_chunks": retrieved_indices_oracle,
        "augmented_prompt": llama_oracle_augmented_prompt,
        "generated_answer": llama_answer_oracle,
    })

print("------------Final Answers (RAG):--------------")
for query, answer in answers_rag.items():
    print(f"Query: {query}\n{answer}\n")

print("------------Final Answers (Oracle):---------------")
for query, answer in answers_oracle.items():
    print(f"Query: {query}\n{answer}\n")

------------Final Answers (RAG):--------------
Query: where is the lady with an ermine located
1st Model RAG Answer: Czartoryski Museum
2nd Model RAG Answer: Czartoryski Museum (Poland)
Real Answer: ['the National Museum in Kraków']

Query: lok sabha or rajya sabha which is bigger
1st Model RAG Answer: Lok Sabha
2nd Model RAG Answer: Lok Sabha is bigger than Rajya Sabha.

Note: This answer is based on the information provided in the three documents. I have used the information from Document 1 to determine the difference in the number of members in Lok Sabha and Rajya Sabha.
Real Answer: ['Lok Sabha']

Query: who was the actress who played pinky tuscadero on happy days
1st Model RAG Answer: Roz Kelly
2nd Model RAG Answer: Roz Kelly (born July 29, 1943) is an American actress, perhaps best known for playing Arthur "Fonzie" Fonzarelli's (Henry Winkler) girlfriend Carol "Pinky" Tuscadero on the
Real Answer: ['Roz Kelly']

Query: who wrote the song i want you back
1st Model RAG Answer: Berr

### JSONL File Generation

Now we generate the JSONL files.

In [ ]:
# Generate JSONL files for each model and pipeline
generate_jsonl_file(t5_baseline_all_results, "all-test", "flan-t5-large", "Baseline", "generated_responses")
generate_jsonl_file(llama_baseline_all_results, "all-test", "llama-3.2-1b-instruct", "Baseline", "generated_responses")
generate_jsonl_file(t5_rag_all_results, "all-test", "flan-t5-large", "RAG", "generated_responses")
generate_jsonl_file(llama_rag_all_results, "all-test", "llama-3.2-1b-instruct", "RAG", "generated_responses")
generate_jsonl_file(t5_oracle_all_results, "all-test", "flan-t5-large", "Oracle", "generated_responses")
generate_jsonl_file(llama_oracle_all_results, "all-test", "llama-3.2-1b-instruct", "Oracle", "generated_responses")

Its_always_loss-all-test-flan-t5-large-Baseline.jsonl generation...
Its_always_loss-all-test-llama-3.2-1b-instruct-Baseline.jsonl generation...
Its_always_loss-all-test-flan-t5-large-RAG.jsonl generation...
Its_always_loss-all-test-llama-3.2-1b-instruct-RAG.jsonl generation...
Its_always_loss-all-test-flan-t5-large-Oracle.jsonl generation...
Its_always_loss-all-test-llama-3.2-1b-instruct-Oracle.jsonl generation...


## Part 2: Evaluation

In this part, we focus on the automatic evaluation process through EM, subEM, and METEOR metrics, and LLM-as-a-Judge annotation, including also the Excel file generation for manual annotation purposes. We then compute the final_evaluation function over the already loaded annotated Excel and JSONL files to visualize the final results of the analysis.

### Evaluation via Standard Metrics

In [ ]:
queries = ds["test"]["query"]
query_ids = ds["test"]["query_id"]
short_answers = ds["test"]["short_answer"]
wikidata_ids = ds["test"]["wikidata_id"]

wikidata_answers = []

In [ ]:
# For each query, find any synonym for the short answer. Then, compute the evaluations based on EM, subEM, METEOR
for query_id, wikidata_id, short_answer in tqdm(zip(query_ids, wikidata_ids, short_answers), total=len(query_ids), desc="Fetching Wikidata Ground Truths"):
    wikidata_answers.append(get_wikidata_ground_truth(wikidata_id, short_answer[0]))

Fetching Wikidata Ground Truths: 100%|██████████| 2000/2000 [36:48<00:00,  1.10s/it]


In [ ]:
evaluate_generations("t5", query_ids, short_answers, wikidata_answers, "Baseline", "rag/all-test/Its_always_loss-all-test-flan-t5-large-Baseline.jsonl")
evaluate_generations("llama", query_ids, short_answers, wikidata_answers, "Baseline", "rag/all-test/Its_always_loss-all-test-llama-3.2-1b-instruct-Baseline.jsonl")

evaluate_generations("t5", query_ids, short_answers, wikidata_answers, "RAG", "rag/all-test/Its_always_loss-all-test-flan-t5-large-RAG.jsonl")
evaluate_generations("llama", query_ids, short_answers, wikidata_answers, "RAG", "rag/all-test/Its_always_loss-all-test-llama-3.2-1b-instruct-RAG.jsonl")

evaluate_generations("t5", query_ids, short_answers, wikidata_answers, "Oracle", "rag/all-test/Its_always_loss-all-test-flan-t5-large-Oracle.jsonl")
evaluate_generations("llama", query_ids, short_answers, wikidata_answers, "Oracle", "rag/all-test/Its_always_loss-all-test-llama-3.2-1b-instruct-Oracle.jsonl")

Average EM for t5 on Baseline: 0.0245
Average EM for t5 on Baseline (using Wikidata answers as ground truth): 0.0280
Average subEM for t5 on Baseline: 0.0350
Average subEM for t5 on Baseline (using Wikidata answers as ground truth): 0.0495
Average METEOR for t5 on Baseline: 0.0646
Average METEOR for t5 on Baseline (using Wikidata answers as ground truth): 0.0714
Average EM for llama on Baseline: 0.0020
Average EM for llama on Baseline (using Wikidata answers as ground truth): 0.0025
Average subEM for llama on Baseline: 0.1060
Average subEM for llama on Baseline (using Wikidata answers as ground truth): 0.1550
Average METEOR for llama on Baseline: 0.0959
Average METEOR for llama on Baseline (using Wikidata answers as ground truth): 0.1036
Average EM for t5 on RAG: 0.3475
Average EM for t5 on RAG (using Wikidata answers as ground truth): 0.3785
Average subEM for t5 on RAG: 0.4505
Average subEM for t5 on RAG (using Wikidata answers as ground truth): 0.5200
Average METEOR for t5 on RAG: 0.

### Evaluation via LLM-as-a-Judge

In [ ]:
set_seed() # Fixing the seed for reproducibility of the LLM-as-a-judge phase

In [ ]:
selected_queries = random.sample(list(zip(queries, query_ids, short_answers)), 250) # Select a random subset of 250 queries for judging
queries, query_ids, short_answers = zip(*selected_queries)

# Define the LLM-as-a-Judge model for flan-t5-large, then compute the judgements and generate: a JSONL file, an Excel file (for manual annotation purposes)
judgements_t5 = judge_model("flan-t5-large", selected_queries)
generate_jsonl_file(judgements_t5, "judge-subset", "flan-t5-large", "RAG", "LLM_judge")
jsonl_path = f"rag/judge-subset/Its_always_loss-judge-subset-flan-t5-large-RAG.jsonl"
export_judge_to_excel(jsonl_path, f"rag/judge-subset/Annotations-flan-t5-large.xlsx", queries, short_answers)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


Judging Answers:   0%|          | 0/250 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Judging Answers: 100%|██████████| 250/250 [23:05<00:00,  5.54s/it]


Its_always_loss-judge-subset-flan-t5-large-RAG.jsonl generation...
excel file saved : rag/judge-subset/Annotations-flan-t5-large.xlsx


In [ ]:
!mkdir -p annotations

url = 'https://drive.google.com/uc?id=1z3M6o9oJDYOfxsT8m2dK5GN3bxxpb7CX'

!gdown {url} -O annotations/Annotations-flan-t5-large.xlsx
parent_dir = os.getcwd()

Downloading...
From: https://drive.google.com/uc?id=1z3M6o9oJDYOfxsT8m2dK5GN3bxxpb7CX
To: /content/annotations/Annotations-flan-t5-large.xlsx
100% 280k/280k [00:00<00:00, 4.59MB/s]


In [ ]:
# convert exel file with annotation into jsonl file and evaluate it
excel_file = os.path.join(parent_dir, "annotations", "Annotations-flan-t5-large.xlsx")
jsonl_file = os.path.join(parent_dir, "rag", "judge-subset", "Its_always_loss-judge-subset-flan-t5-large-RAG.jsonl")

export_annotations(excel_file, jsonl_file)
final_evaluation(jsonl_file)

jsonl file saved : /content/rag/judge-subset/Its_always_loss-judge-subset-flan-t5-large-RAG.jsonl
Average model's score for annotator 1: 0.7440
Average model's score for annotator 2: 0.7440
Average model's score for LLM-as-a-Judge: 0.7480
Cohen's Kappa between annotators: 1.0000
Cohen's Kappa between annotator 1 and LLM-as-a-Judge: 0.6728
Cohen's Kappa between annotator 2 and LLM-as-a-Judge: 0.6728


## Part 3: Generate answer on blind set

This part is focused only on the blind set generation for RAG system models as requested by the guidelines.

In [ ]:
def blind_generation_pipeline(ds, t5_model, t5_tokenizer, t5_device, llama_model, llama_tokenizer, llama_device, all_mini_jsonl):
    blind_queries = ds["blind"]["query"]
    answers_rag_blind = {}
    t5_rag_all_results_blind = []
    llama_rag_all_results_blind = []
    for query in blind_queries[:5]:#tqdm(blind_queries, desc="RAG Pipeline Procesing on blind set"):
        item = ds["blind"][blind_queries.index(query)]
        query = item["query"]
        query_id = item["query_id"]
        candidate = item["candidate_chunks"]
        wikidata_id = item["wikidata_id"]

        wikidata_info = get_wikidata_entity(wikidata_id)

        retrieved_chunks_blind, retrieved_indices_blind = get_top_k_chunks(query_id, all_mini_jsonl, candidate, k=3)

        t5_answer_rag_blind, t5_augmented_prompt_blind = rag(t5_model, t5_tokenizer, query, wikidata_info, retrieved_chunks_blind, t5_device)
        llama_answer_rag_blind, llama_augmented_prompt_blind = rag(llama_model, llama_tokenizer, query, wikidata_info, retrieved_chunks_blind, llama_device)

        answers_rag_blind[query] = f"1st Model RAG Answer: {t5_answer_rag_blind}\n2nd Model RAG Answer: {llama_answer_rag_blind}\n"

        t5_rag_all_results_blind.append({
            "query_id": query_id,
            "retrieved_chunks": retrieved_indices_blind,
            "augmented_prompt": t5_augmented_prompt_blind,
            "generated_answer": t5_answer_rag_blind,
        })

        llama_rag_all_results_blind.append({
            "query_id": query_id,
            "retrieved_chunks": retrieved_indices_blind,
            "augmented_prompt": llama_augmented_prompt_blind,
            "generated_answer": llama_answer_rag_blind,
        })

    print("------------Final Answers (RAG):--------------")
    for query, answer in answers_rag_blind.items():
        print(f"Query: {query}\n{answer}\n")

    generate_jsonl_file(t5_rag_all_results_blind, "blind", "flan-t5-large", "RAG", "generated_responses")
    generate_jsonl_file(llama_rag_all_results_blind, "blind", "llama-3.2-1b-instruct", "RAG", "generated_responses")

In [ ]:
all_mini_jsonl_blind = os.path.join(PREDICTIONS_DIR, "blind", "Its_always_loss-blind-all-miniLM-L6-v2-2-mnr-cosine.jsonl")
blind_generation_pipeline(ds, t5_model, t5_tokenizer, t5_device, llama_model, llama_tokenizer, llama_device, all_mini_jsonl_blind)


NameError: name 'get_top_k_chunks' is not defined